# Workshop 1 - Data Preprocessing and Supervised Learning


## Contexto y enfoque

Este trabajo se apoyará en los notebooks de referencia `MIAX ML - 01 Data Preprocessing.ipynb` y `MIAX ML - 03 Linear Regression.ipynb`.

El dataset base serán `navs.pickle`.
Se prioriza simplicidad, claridad y justificación explícita de cada decisión metodológica.

## 1. Objetivo y tesis financiera

### Hipótesis financiera
Como gestores de un fondo de fondos, planteamos la hipótesis de que **los fondos con mejor comportamiento histórico reciente, especialmente los expuestos a Asia, tenderán a mantener un desempeño relativo superior en el siguiente horizonte de inversión**, siempre que se combine este sesgo con diversificación para controlar riesgo idiosincrático.

Esta tesis se apoya en dos ideas conocidas en finanzas:
1. **Persistencia parcial del momentum**: activos/fondos con buen desempeño reciente pueden seguir mostrando fortaleza durante un tiempo.
2. **Sesgo temático con diversificación**: sobreponderar una región con tesis macro (Asia) puede capturar una oportunidad estructural, pero debe combinarse con otros fondos para evitar concentración excesiva.

### Justificación económica y evidencia previa
- En la literatura financiera existe evidencia de **momentum** en distintos mercados y horizontes intermedios, aunque no es estable en todos los periodos.
- En fondos, la persistencia de rentabilidades suele ser **más débil que en acciones individuales** y puede verse afectada por comisiones, cambios de gestor y rotación de estilo.
- Por tanto, la hipótesis es razonable, pero debe validarse empíricamente con controles de robustez y evitando sesgos de selección/look-ahead.

### Objetivo principal
El objetivo principal será una **clasificación supervisada** orientada a decisión de inversión:
- Predecir si un fondo **superará un umbral de rentabilidad de mercado** en el horizonte futuro definido.
- La salida del modelo se utilizará para apoyar decisiones de **comprar, mantener/no operar, reducir/vender** y **ajustar exposición**.

> Nota: aunque una regresión de retorno futuro es viable, en este caso la clasificación es más alineada con la decisión real (supera/no supera benchmark).

### Variable objetivo (target)
Definimos, para cada fondo \(i\) y fecha \(t\), el retorno futuro a horizonte \(h\):
\[
R_{i,t \rightarrow t+h} = \frac{NAV_{i,t+h}}{NAV_{i,t}} - 1
\]

Sea \(R^{mkt}_{t \rightarrow t+h}\) el retorno del mercado de referencia en el mismo horizonte, y \(\delta\) un umbral mínimo (alpha objetivo).  
Entonces:
\[
y_{i,t}=1 \quad \text{si} \quad R_{i,t \rightarrow t+h} - R^{mkt}_{t \rightarrow t+h} > \delta
\]
\[
y_{i,t}=0 \quad \text{en caso contrario}
\]

Esto convierte el problema en clasificación binaria de “outperformers”.

### Horizonte temporal de predicción
Se evaluarán dos horizontes candidatos:
- **1 mes**: más observaciones y capacidad de reacción, pero mayor ruido.
- **1 año**: más alineado con construcción estratégica de cartera (fondo de fondos), pero menor tamaño muestral y menor reactividad.

Como hipótesis base para esta práctica, se prioriza **1 año** por coherencia con el mandato de asignación estratégica, y se deja **1 mes** como análisis de sensibilidad.

### Restricciones y alcance de datos
- Universo: fondos del archivo pickle, con campos disponibles: `allfunds_id`, `isin`, `name`, `nav`.
- Periodo observado: **2016-01-05 a 2021-07-16**.
- Restricción operativa principal: seleccionar fondos que estén disponibles/“existentes” en la fecha de decisión (rol asesor a 2021-07-16), minimizando sesgo de supervivencia en la medida posible con la información disponible.
- Frecuencia de NAV heterogénea entre fondos; será necesario homogeneizar frecuencia antes del modelado.

### Prevención de look-ahead bias
Para evitar fuga de información temporal:
- Construir variables explicativas únicamente con información disponible en \(t\).
- Etiquetar el target con retornos futuros \((t, t+h)\), sin usar datos posteriores en features.
- Aplicar validación temporal (entrenar en pasado, validar/test en periodos posteriores), evitando mezclas aleatorias.
- En selección de universo por fecha, usar solo el “estante” observable en cada corte temporal, no información del dataset completo a futuro.

### Métricas candidatas y criterio de evaluación
Como aún no se fija una métrica final, se considerarán:
- **ROC-AUC**: buena visión global de ranking; menos intuitiva para decisión final con umbral.
- **PR-AUC**: útil si hay desbalance de clases (outperformers escasos).
- **Precision**: calidad de señales de compra; puede sacrificar cobertura.
- **Recall**: captura más oportunidades; puede introducir más falsos positivos.
- **F1-score**: equilibrio entre precision y recall.
- **Balanced Accuracy**: robusta ante clases desbalanceadas.
- **MCC**: métrica global robusta en binario desbalanceado.
- **Métricas económicas ex post** (retorno cartera, volatilidad, drawdown, Sharpe): imprescindibles para validar utilidad financiera real.


## 2. Carga y alineación temporal de datos

En esta sección se construye una base temporalmente consistente a partir de `navs.pickle` (diccionario de fondos).

**Criterios operativos acordados**
- Frecuencia objetivo: `W-FRI`.
- Valor semanal representativo: último NAV disponible de la semana (`last`).
- Duplicados por fecha dentro de un fondo: conservar última observación (`keep='last'`).
- Rango final: intersección temporal común entre todos los fondos válidos.
- Imputación de NAV tras remuestreo: `ffill(limit=5)`.
- Reporte explícito de faltantes antes y después de imputar.
- Señalización de fondos con temporalidad detectada `weekly` o `monthly`.

**Nota de diseño (pendiente de decisión posterior)**
- El umbral de exclusión por histórico mínimo (p. ej., nº mínimo de semanas) se deja parametrizado pero no se activa en esta fase.

**Entregable de la sección:**
- Dataset semanal alineado y limpio en formato largo.
- Tablas de control de calidad temporal y cobertura.
- Archivo parquet final reproducible en cada ejecución.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

# =========================
# Configuración
# =========================
PICKLE_PATH = Path("DOCS_CLASE") / "MachineLearning" / "dataset" / "navs.pickle"
OUTPUT_DIR = Path("data") / "processed"
OUTPUT_PARQUET = OUTPUT_DIR / "navs_weekly_aligned.parquet"

WEEK_FREQ = "W-FRI"          # criterio acordado
FFILL_LIMIT = 5               # máximo de periodos consecutivos a rellenar
MIN_WEEKS_REQUIRED = None     # pendiente de decisión (no filtra en esta fase)


# =========================
# Funciones auxiliares
# =========================
def load_navs_pickle(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")
    obj = pd.read_pickle(path)
    if not isinstance(obj, dict):
        raise TypeError(f"Se esperaba dict y se recibió: {type(obj)}")
    return obj


def classify_frequency_from_dates(dates: pd.Series) -> str:
    """Clasifica la frecuencia observada por mediana de separación entre fechas."""
    if dates is None or dates.empty:
        return "invalid"
    d = dates.dropna().sort_values().drop_duplicates()
    if len(d) < 3:
        return "insufficient_data"

    gaps = d.diff().dropna().dt.days
    if gaps.empty:
        return "insufficient_data"

    med_gap = float(gaps.median())

    # Heurística práctica
    if med_gap <= 2:
        return "daily"
    if 5 <= med_gap <= 9:
        return "weekly"
    if 25 <= med_gap <= 35:
        return "monthly"
    return "irregular"


def standardize_single_fund(fund_id, df):
    """
    Limpia y estandariza un fondo individual conservando columnas originales.
    Requisitos esperados: columna date + nav.
    """
    required_cols = {"date", "nav"}
    out = {
        "fund_id": fund_id,
        "status": "ok",
        "reason": "",
        "n_raw": 0,
        "n_after_parse": 0,
        "n_nat_dates": 0,
        "n_dup_dates": 0,
        "frequency_detected": "invalid",
        "min_date": pd.NaT,
        "max_date": pd.NaT,
        "n_obs": 0,
    }

    if not isinstance(df, pd.DataFrame):
        out["status"] = "discarded"
        out["reason"] = "not_dataframe"
        return None, out

    out["n_raw"] = len(df)

    missing = required_cols.difference(df.columns)
    if missing:
        out["status"] = "discarded"
        out["reason"] = f"missing_cols:{sorted(missing)}"
        return None, out

    tmp = df.copy()

    # Parse robusto de fecha: fuerza UTC para homogeneizar, luego elimina timezone
    tmp["date"] = pd.to_datetime(tmp["date"], errors="coerce", utc=True)
    out["n_nat_dates"] = int(tmp["date"].isna().sum())
    tmp = tmp.dropna(subset=["date"]).copy()

    if tmp.empty:
        out["status"] = "discarded"
        out["reason"] = "all_dates_nat"
        return None, out

    tmp["date"] = tmp["date"].dt.tz_convert("UTC").dt.tz_localize(None)

    # Orden cronológico
    tmp = tmp.sort_values("date").reset_index(drop=True)

    # Duplicados por fecha: se conserva la última observación
    out["n_dup_dates"] = int(tmp.duplicated(subset=["date"], keep="last").sum())
    tmp = tmp.drop_duplicates(subset=["date"], keep="last").copy()

    # Cast de nav a numérico para robustez
    tmp["nav"] = pd.to_numeric(tmp["nav"], errors="coerce")

    out["n_after_parse"] = len(tmp)
    out["frequency_detected"] = classify_frequency_from_dates(tmp["date"])
    out["min_date"] = tmp["date"].min()
    out["max_date"] = tmp["date"].max()
    out["n_obs"] = len(tmp)

    return tmp, out


def to_weekly_last(df: pd.DataFrame, week_freq: str = "W-FRI") -> pd.DataFrame:
    """Remuestrea un fondo a semanal usando último NAV/valor disponible de cada semana."""
    wk = (
        df.set_index("date")
          .sort_index()
          .resample(week_freq)
          .last()
          .reset_index()
    )
    return wk


# =========================
# 1) Carga y validación del dict
# =========================
navs_dict = load_navs_pickle(PICKLE_PATH)
print(f"Fondos totales en pickle: {len(navs_dict):,}")


# =========================
# 2) Limpieza por fondo + metadata de calidad
# =========================
clean_funds = {}
quality_rows = []

for fund_id, df in navs_dict.items():
    cleaned_df, meta = standardize_single_fund(fund_id, df)
    quality_rows.append(meta)
    if meta["status"] == "ok":
        clean_funds[fund_id] = cleaned_df

quality_df = pd.DataFrame(quality_rows)
summary_status = quality_df.groupby(["status", "reason"], dropna=False).size().reset_index(name="n_funds")

print("\nResumen general de fondos válidos vs descartados")
print(summary_status.sort_values(["status", "n_funds"], ascending=[True, False]).head(20))


# =========================
# 3) Distribución de frecuencia detectada y marcaje semanal/mensual
# =========================
freq_dist = (
    quality_df.loc[quality_df["status"] == "ok", "frequency_detected"]
    .value_counts(dropna=False)
    .rename_axis("frequency_detected")
    .reset_index(name="n_funds")
)
print("\nDistribución de frecuencias detectadas")
print(freq_dist)

quality_df["is_weekly_or_monthly"] = quality_df["frequency_detected"].isin(["weekly", "monthly"])


# =========================
# 4) Remuestreo semanal + reporte faltantes antes/después de ffill(limit=5)
# =========================
weekly_funds = {}
missing_report_rows = []
coverage_rows = []

for fund_id, df in clean_funds.items():
    wk = to_weekly_last(df, week_freq=WEEK_FREQ)

    missing_before = int(wk["nav"].isna().sum())
    wk["nav"] = wk["nav"].ffill(limit=FFILL_LIMIT)
    missing_after = int(wk["nav"].isna().sum())

    # Rehidrata columnas estáticas si quedaron vacías tras resample
    # (allfunds_id, isin, name suelen ser constantes por fondo)
    static_cols = [c for c in ["allfunds_id", "isin", "name"] if c in wk.columns]
    for c in static_cols:
        wk[c] = wk[c].ffill().bfill()

    # Añade fund_id explícito por trazabilidad
    wk["fund_id"] = fund_id

    # Filtro opcional no activado (decisión pendiente)
    if (MIN_WEEKS_REQUIRED is not None) and (len(wk) < MIN_WEEKS_REQUIRED):
        continue

    weekly_funds[fund_id] = wk

    missing_report_rows.append({
        "fund_id": fund_id,
        "missing_nav_before_ffill": missing_before,
        "missing_nav_after_ffill": missing_after,
        "filled_points": missing_before - missing_after,
    })

    coverage_rows.append({
        "fund_id": fund_id,
        "min_date": wk["date"].min(),
        "max_date": wk["date"].max(),
        "n_obs_weekly": len(wk),
    })

missing_report_df = pd.DataFrame(missing_report_rows)
coverage_df = pd.DataFrame(coverage_rows)

print("\nControl de faltantes NAV (antes vs después de ffill)")
print(missing_report_df[["missing_nav_before_ffill", "missing_nav_after_ffill", "filled_points"]].sum().to_frame("total").T)


# =========================
# 5) Rango temporal común (intersección) + alineación final
# =========================
if coverage_df.empty:
    raise RuntimeError("No hay fondos semanales válidos para alinear.")

common_start = coverage_df["min_date"].max()
common_end = coverage_df["max_date"].min()

if pd.isna(common_start) or pd.isna(common_end) or common_start > common_end:
    raise RuntimeError("No existe intersección temporal válida entre los fondos seleccionados.")

print("\nRango temporal común elegido")
print(f"Inicio común: {common_start.date()} | Fin común: {common_end.date()}")
print("Justificación práctica: usar fechas comparables para todos los fondos evita mezclar ventanas con distinta información.")

aligned_list = []
size_before = 0

for fund_id, wk in weekly_funds.items():
    size_before += len(wk)
    aligned = wk[(wk["date"] >= common_start) & (wk["date"] <= common_end)].copy()
    aligned_list.append(aligned)

final_aligned_df = pd.concat(aligned_list, axis=0, ignore_index=True)

# Orden final para análisis
final_aligned_df = final_aligned_df.sort_values(["fund_id", "date"]).reset_index(drop=True)

size_after = len(final_aligned_df)
print("\nTamaño antes/después de alinear")
print(f"Filas antes (sumando fondos semanales): {size_before:,}")
print(f"Filas después (rango común):            {size_after:,}")


# =========================
# 6) Resumen final de control
# =========================
n_total = len(navs_dict)
n_valid = len(clean_funds)
n_weekly_final = len(weekly_funds)
n_discarded = n_total - n_valid

print("\nResumen final")
print(f"Fondos totales:               {n_total:,}")
print(f"Fondos válidos tras limpieza: {n_valid:,}")
print(f"Fondos descartados:           {n_discarded:,}")
print(f"Fondos en dataset semanal:    {n_weekly_final:,}")


# =========================
# 7) Guardado reproducible a parquet (reemplaza si existe)
# =========================
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if OUTPUT_PARQUET.exists():
    os.remove(OUTPUT_PARQUET)

final_aligned_df.to_parquet(OUTPUT_PARQUET, index=False)
print(f"\nArchivo guardado: {OUTPUT_PARQUET.as_posix()}")


# =========================
# 8) Tablas de salida útiles para la memoria
# =========================
# - summary_status: válidos vs descartados y motivo
# - freq_dist: distribución de frecuencias detectadas
# - quality_df: detalle por fondo (incluye flag weekly/monthly)
# - missing_report_df: faltantes NAV antes y después de ffill
# - coverage_df: cobertura temporal por fondo (semanal)
# - final_aligned_df: dataset final alineado (largo)


### Texto para memoria (sección 2)

Se cargó `navs.pickle` como diccionario de fondos y se validó que cada entrada tuviera estructura tabular mínima (`date`, `nav`). La limpieza temporal se centró en cuatro puntos prácticos: parseo robusto de fechas (con `errors='coerce'`), eliminación de fechas inválidas (`NaT`), orden cronológico y resolución explícita de duplicados por fecha conservando la última observación (`keep='last'`).

Para homogeneizar series con distinta granularidad, se detectó la frecuencia observada por fondo (diaria, semanal, mensual o irregular) usando la separación temporal típica entre observaciones. Además, se marcó de forma explícita qué fondos muestran temporalidad semanal o mensual para facilitar controles posteriores del universo.

La frecuencia final de trabajo se fijó en semanal (`W-FRI`) y el valor representativo de cada semana se definió como el último NAV disponible (`last`). Este criterio reduce ruido frente a frecuencia diaria, mantiene suficiente resolución para análisis comparativo y es coherente con decisiones de asignación no intradía.

Tras el remuestreo, se aplicó imputación hacia delante sobre `nav` con `ffill(limit=5)`, limitando el relleno a huecos cortos para no propagar valores en periodos largos sin información. Se reportaron faltantes antes y después de imputar para cuantificar el impacto real del tratamiento.

La alineación final se hizo con intersección temporal común entre fondos válidos (mismo rango de fechas para todos), priorizando comparabilidad directa entre series y evitando sesgos por ventanas desiguales. Como contrapartida, este enfoque puede reducir longitud histórica útil; por ello, el umbral de histórico mínimo por fondo queda parametrizado para decidirse en la siguiente fase.

## 3. EDA (exploración de calidad)

En esta parte se revisará calidad de datos de forma inicial: tamaños, tipos, nulos y comportamientos básicos.
El objetivo es detectar problemas antes de limpiar y modelar.

**Decisiones a justificar**
- Qué indicadores de calidad son prioritarios.
- Qué gráficos mínimos aportan más información.
- Qué umbrales preliminares se usarán para alertas.

**Entregable de la sección:** resumen de calidad con tablas y gráficos básicos de diagnóstico.

In [ ]:
# TODO: Revisar shape, dtypes y nulos
# TODO: Generar visualizaciones básicas de calidad de datos

## 4. Limpieza y preprocesado

Aquí se aplicarán reglas explícitas para tratar faltantes, duplicados y valores inconsistentes.
La intención es mejorar la calidad sin introducir sesgos innecesarios.

**Decisiones a justificar**
- Umbral para excluir fondos con demasiados faltantes.
- Método de imputación elegido (si aplica).
- Criterio para tratar valores no positivos y outliers.

**Entregable de la sección:** versión limpia del dataset con reglas documentadas.

In [ ]:
# TODO: Definir y aplicar reglas de limpieza
# TODO: Guardar tablas resumen antes/después de limpiar

## 5. Feature engineering

En este bloque se transformarán NAVs y factores en variables listas para modelado supervisado.
Se buscará un conjunto de features simple, interpretable y temporalmente correcto.

**Decisiones a justificar**
- Definición de retornos y transformaciones aplicadas.
- Variables predictoras elegidas (actuales y/o rezagadas).
- Definición final de `X` y `y`.

**Entregable de la sección:** matriz de features y variable objetivo preparadas para train/test.

In [ ]:
# TODO: Construir features (retornos, rezagos, variables seleccionadas)
# TODO: Definir variable objetivo para supervisado

## 6. Modelado supervisado

Aquí se entrenarán modelos supervisados simples para evaluar la señal predictiva de las features.
La prioridad será comparar de forma clara y con supuestos controlados.

**Decisiones a justificar**
- Modelo principal seleccionado y por qué.
- Configuración mínima de entrenamiento.
- Cómo se evita complejidad innecesaria.

**Entregable de la sección:** modelo entrenado y predicciones sobre el conjunto de test.

In [ ]:
# TODO: Crear split temporal train/test sin shuffle
# TODO: Entrenar modelo supervisado base y generar predicciones

## 7. Evaluación y visualización

Esta sección medirá el rendimiento del modelo con métricas adecuadas al objetivo elegido.
También se incluirán visualizaciones simples para interpretar los resultados.

**Decisiones a justificar**
- Métricas principales seleccionadas.
- Qué gráficos se usarán para interpretar errores/aciertos.
- Cómo se comparará el resultado con una referencia básica.

**Entregable de la sección:** tabla de métricas y gráficos de evaluación del modelo.

In [ ]:
# TODO: Calcular métricas de evaluación según el tipo de objetivo
# TODO: Crear gráficos simples de resultados (real vs predicho o matriz de confusión)

## 8. Conclusiones y limitaciones

Finalmente, se resumirán hallazgos clave, limitaciones metodológicas y riesgos del enfoque.
El objetivo es cerrar con una interpretación honesta y accionable.

**Decisiones a justificar**
- Qué hipótesis quedó mejor respaldada.
- Qué límites tienen datos y modelo.
- Qué mejoras serían prioritarias en una siguiente iteración.

**Entregable de la sección:** conclusión ejecutiva y lista de limitaciones/siguientes pasos.

In [ ]:
# TODO: Redactar conclusiones finales basadas en resultados
# TODO: Documentar limitaciones y próximos pasos

## Checklist de ejecución

- [ ] 1. Objetivo y tesis financiera
- [ ] 2. Carga y alineación temporal de datos
- [ ] 3. EDA (exploración de calidad)
- [ ] 4. Limpieza y preprocesado
- [ ] 5. Feature engineering
- [ ] 6. Modelado supervisado
- [ ] 7. Evaluación y visualización
- [ ] 8. Conclusiones y limitaciones

## Notas de reproducibilidad

- Trabajar siempre con rutas relativas del proyecto.
- Evitar data leakage en cualquier transformación.
- Usar split temporal (sin `shuffle`) para train/test.
- Comentar celdas con claridad para trazabilidad académica.